<a href="https://colab.research.google.com/github/taticaires/Lab01_Aula04_OpenCV_SciPy/blob/main/C%C3%B3pia_de_Lab01_Aula04_OpenCV_SciPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratório 01 — Visão computacional com OpenCV e SciPy
### Processamento de Imagem e Sinais — Tecnólogo em Inteligência Artificial
**FMU — 2026.2** · Prof. Luciano Tadeu Pereira · Aula 04 — 11/09/2026

---
Você vai construir o conjunto **PulmoScan FMU** e o módulo de pré-processamento do projeto. O `pulmoscan_meta.csv` gerado aqui é insumo das aulas 06, 08, 10, 13 e 15. **Guarde-o.**

> **Como usar:** rode as células **em ordem**. Onde houver `# SUA VEZ`, escreva o código antes de avançar.
> Onde houver **Responda:**, escreva a resposta na própria célula de texto.

---

**ATENÇÃO**: Antes de começar, faça uma cópia deste notebook para sua conta do Google Drive. Assim, você poderá editar e testar sem modificar o original.

Para isso, clique em "Arquivo" → "Salvar uma cópia no Drive".

OBS: Para executar este notebook, **não** é necessário um ambiente de execução com GPU.

---


## 1. O conjunto de dados

O plano de ensino indica o *NIH Chest X-Ray Dataset* para o projeto de vocês. Aqui usamos um conjunto **sintético**: imagens geradas por código, com anatomia esquemática e nódulos simulados. Isso é deliberado — permite exercitar o pipeline inteiro sem tocar em dado de saúde real e com o gabarito perfeitamente conhecido. **O que você aprender aqui transfere; os números, não.** No Artefato 2 vocês usarão o dataset real do grupo.

In [ ]:
# === PulmoScan FMU — gerador do conjunto sintético =========================
# Estas NÃO são radiografias reais. São imagens sintéticas, geradas por código,
# com anatomia esquemática e nódulos simulados. Nenhum dado de saúde real é usado.
# Rode esta célula uma vez por sessão do Colab.
import os, csv, zlib, struct
import numpy as np

TAM = 256

def _anatomia(rng):
    """Assinatura anatômica do paciente — não muda entre os dois exames dele."""
    return dict(larg=rng.uniform(100,116), alt=rng.uniform(112,128),
                pulm_l=rng.uniform(31,37), pulm_a=rng.uniform(57,66),
                sep=rng.uniform(42,50), coracao=rng.uniform(30,38),
                costela=rng.uniform(7.3,7.6), base=rng.uniform(86,98),
                densidade=rng.uniform(58,66))

def _fundo(a):
    y, x = np.mgrid[0:TAM,0:TAM].astype(np.float32); cx = cy = TAM/2
    img = np.full((TAM,TAM), a["base"], np.float32)
    torax = ((x-cx)/a["larg"])**2 + ((y-cy)/a["alt"])**2
    img += 46*np.clip(1.35-torax,0,1)
    for sx in (-1,1):
        d = ((x-(cx+sx*a["sep"]))/a["pulm_l"])**2 + ((y-(cy-6))/a["pulm_a"])**2
        img -= a["densidade"]*np.clip(1.0-d,0,1)**0.75
    img += 40*np.exp(-(((x-cx)/9.0)**2))
    dc = ((x-(cx+14))/a["coracao"])**2 + ((y-(cy+34))/30.0)**2
    img += 36*np.clip(1.0-dc,0,1)**0.6
    img += 8*np.sin(y/a["costela"] + 0.55*np.cos(x/46.0))*np.clip(1.3-torax,0,1)
    img *= 1.0 - 0.20*(((x-cx)/TAM)**2 + ((y-cy)/TAM)**2)*3.2
    return img

def _mascara(a):
    y, x = np.mgrid[0:TAM,0:TAM].astype(np.float32); cx = cy = TAM/2
    m = np.zeros((TAM,TAM), bool)
    for sx in (-1,1):
        d = ((x-(cx+sx*a["sep"]))/(a["pulm_l"]-4))**2 + ((y-(cy-6))/(a["pulm_a"]-6))**2
        m |= d < 1.0
    return m

_F = {"0":["01110","10001","10011","10101","11001","10001","01110"],
      "1":["00100","01100","00100","00100","00100","00100","01110"],
      "2":["01110","10001","00001","00110","01000","10000","11111"],
      "3":["11111","00010","00100","00010","00001","10001","01110"],
      "4":["00010","00110","01010","10010","11111","00010","00010"],
      "5":["11111","10000","11110","00001","00001","10001","01110"],
      "6":["00110","01000","10000","11110","10001","10001","01110"],
      "7":["11111","00001","00010","00100","01000","01000","01000"],
      "8":["01110","10001","10001","01110","10001","10001","01110"],
      "9":["01110","10001","10001","01111","00001","00010","01100"],
      "P":["11110","10001","10001","11110","10000","10000","10000"],
      "T":["11111","00100","00100","00100","00100","00100","00100"],
      "-":["00000","00000","00000","11111","00000","00000","00000"]}

def _gravar_id(img, texto):
    """Identificação 'burned-in': o ID do paciente escrito NOS PIXELS da imagem."""
    x0, y0 = 6, 6
    for ch in texto:
        for r, linha in enumerate(_F.get(ch, ["00000"]*7)):
            for c, b in enumerate(linha):
                if b == "1":
                    img[y0+r*2:y0+r*2+2, x0+c*2:x0+c*2+2] = 250
        x0 += 12
    return img

def _png(caminho, arr):
    h, w = arr.shape
    raw = b"".join(b"\x00" + arr[i].tobytes() for i in range(h))
    def ch(t, d):
        return struct.pack(">I", len(d)) + t + d + struct.pack(">I", zlib.crc32(t+d) & 0xFFFFFFFF)
    open(caminho, "wb").write(b"\x89PNG\r\n\x1a\n"
        + ch(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 0, 0, 0, 0))
        + ch(b"IDAT", zlib.compress(raw, 6)) + ch(b"IEND", b""))

def gerar_pulmoscan(destino="pulmoscan", n_pacientes=120, seed=42):
    os.makedirs(destino, exist_ok=True)
    rng = np.random.default_rng(seed); linhas = []
    faixas = rng.choice(["pediatrico","adulto","idoso"], n_pacientes, p=[0.20,0.55,0.25])
    aparelhos = rng.choice(["A","B"], n_pacientes, p=[0.65,0.35])
    doentes = rng.random(n_pacientes) < 0.38
    for p in range(n_pacientes):
        pid = f"P{p+1:04d}"; faixa = faixas[p]; apar = aparelhos[p]; doente = bool(doentes[p])
        a = _anatomia(rng); ys, xs = np.nonzero(_mascara(a))
        for e in range(2):
            img = _fundo(a).copy(); raio = contraste = 0.0
            if doente:
                k = rng.integers(len(xs)); nx, ny = int(xs[k]), int(ys[k])
                if faixa == "pediatrico":  raio, contraste = rng.uniform(1.3,1.9), rng.uniform(16,30)
                elif faixa == "idoso":     raio, contraste = rng.uniform(2.0,3.0), rng.uniform(42,72)
                else:                      raio, contraste = rng.uniform(1.8,2.7), rng.uniform(34,62)
                yy, xx = np.mgrid[0:TAM,0:TAM].astype(np.float32)
                img += contraste*np.exp(-(((xx-nx)**2+(yy-ny)**2)/(2*raio**2)))
            img = img + rng.normal(0, 9.0 if apar == "B" else 5.5, img.shape)
            if apar == "B":
                sp = rng.random(img.shape); img[sp < 0.0016] = 255; img[sp > 0.9984] = 0
            img = np.clip(img, 0, 255).astype(np.uint8)
            gravado = (p+e) % 4 == 0
            if gravado:
                img = np.clip(_gravar_id(img.astype(np.float32), f"PT-{p+1:04d}"), 0, 255).astype(np.uint8)
            _png(os.path.join(destino, f"{pid}_E{e+1}.png"), img)
            linhas.append(dict(arquivo=f"{pid}_E{e+1}.png", id_paciente=pid, exame=e+1,
                               faixa_etaria=faixa, aparelho=apar, rotulo=int(doente),
                               raio_nodulo=round(raio,2), contraste_nodulo=round(contraste,1),
                               id_gravado=int(gravado)))
    with open("pulmoscan_meta.csv","w",newline="",encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(linhas[0].keys())); w.writeheader(); w.writerows(linhas)
    return linhas

if not os.path.exists("pulmoscan_meta.csv"):
    L = gerar_pulmoscan()
    print(f"gerado: {len(L)} imagens de 120 pacientes")
else:
    print("pulmoscan_meta.csv já existe — nada a fazer")

## 2. Carregar e inspecionar — o que fazer com toda imagem nova

In [ ]:
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

meta = pd.read_csv("pulmoscan_meta.csv")
print("registros:", len(meta), " pacientes:", meta.id_paciente.nunique())
print(meta.head(4).to_string(index=False))

img = cv2.imread("pulmoscan/P0003_E1.png", cv2.IMREAD_GRAYSCALE)

# As três perguntas que toda imagem responde antes de qualquer processamento:
print("\ndimensões (altura, largura):", img.shape)
print("tipo de dado:", img.dtype, "-> profundidade de", img.dtype.itemsize*8, "bits por pixel")
print("faixa de intensidade: min =", img.min(), " max =", img.max(), " média =", round(img.mean(),1))

> **Atenção ao `None`.** `cv2.imread` não levanta exceção quando o arquivo não existe: devolve `None`. Uma função que não checa isso quebra três linhas depois, com uma mensagem que não ajuda em nada.

In [ ]:
def carregar(caminho):
    """Carrega uma imagem em tons de cinza, falhando de forma explícita.

    Args:
        caminho (str): caminho do arquivo de imagem.
    Returns:
        numpy.ndarray: matriz 2D uint8.
    Raises:
        FileNotFoundError: se o arquivo não existir ou não for uma imagem válida.
    """
    img = cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"não foi possível ler a imagem: {caminho}")
    return img

try:
    carregar("pulmoscan/NAO_EXISTE.png")
except FileNotFoundError as e:
    print("erro tratado corretamente:", e)

**Responda:** por que uma exceção explícita é preferível a devolver `None` silenciosamente neste projeto?



## 3. Histograma — a assinatura da imagem

O histograma diz se o exame está sub ou superexposto **antes** de qualquer algoritmo rodar.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].imshow(img, cmap="gray", vmin=0, vmax=255); ax[0].set_title("P0003_E1"); ax[0].axis("off")
ax[1].hist(img.ravel(), bins=64, range=(0,255), color="#ED0203")
ax[1].set_title("histograma"); ax[1].set_xlabel("intensidade")

escura = np.clip(img.astype(np.int16) - 55, 0, 255).astype(np.uint8)   # subexposição simulada
ax[2].hist(escura.ravel(), bins=64, range=(0,255), color="#3A0604")
ax[2].set_title("mesma imagem subexposta"); ax[2].set_xlabel("intensidade")
plt.tight_layout(); plt.show()

print("original  -> média %.1f  desvio %.1f" % (img.mean(), img.std()))
print("subexposta-> média %.1f  desvio %.1f" % (escura.mean(), escura.std()))

## 4. Espaços de cor e a pegadinha do BGR

O OpenCV carrega em **BGR**; o matplotlib espera **RGB**. Trocar os dois é o erro nº 1 de quem começa.

In [ ]:
colorida = cv2.applyColorMap(img, cv2.COLORMAP_BONE)      # matriz BGR de 3 canais
print("shape:", colorida.shape, "-> (altura, largura, canais)")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].imshow(colorida); ax[0].set_title("ERRADO: BGR exibido como RGB")
ax[1].imshow(cv2.cvtColor(colorida, cv2.COLOR_BGR2RGB)); ax[1].set_title("CERTO: convertido para RGB")
hsv = cv2.cvtColor(colorida, cv2.COLOR_BGR2HSV)
ax[2].imshow(hsv[:,:,2], cmap="gray"); ax[2].set_title("canal V (intensidade) do HSV")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

**Responda:** por que o canal V do HSV se parece tanto com a imagem em tons de cinza original?



## 5. Filtragem espacial — comparando quatro máscaras

O aparelho **B** do nosso conjunto injeta ruído sal-e-pimenta. Vamos ver qual filtro lida com ele.

In [ ]:
ruidosa = carregar("pulmoscan/" + meta[meta.aparelho=="B"].arquivo.iloc[0])

filtros = {
    "original":   ruidosa,
    "média 5x5":  cv2.blur(ruidosa, (5,5)),
    "gaussiano":  cv2.GaussianBlur(ruidosa, (5,5), 0),
    "mediana 3x3":cv2.medianBlur(ruidosa, 3),
}
fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a, (nome, im) in zip(ax, filtros.items()):
    a.imshow(im, cmap="gray", vmin=0, vmax=255); a.set_title(nome); a.axis("off")
plt.tight_layout(); plt.show()

# critério objetivo: quantos pixels extremos (0 ou 255) sobraram?
for nome, im in filtros.items():
    extremos = int(((im == 0) | (im == 255)).sum())
    print(f"{nome:12s} pixels extremos = {extremos:5d}   desvio = {im.std():.2f}")

**Responda:** a mediana e a média removeram a mesma quantidade de pixels extremos. Por que, ainda assim, a mediana é a escolha correta para sal-e-pimenta em imagem médica?



## 6. Detecção de bordas: Sobel e Laplaciano

In [ ]:
sup = cv2.GaussianBlur(img, (3,3), 0)
gx = cv2.Sobel(sup, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(sup, cv2.CV_64F, 0, 1, ksize=3)
mag = cv2.magnitude(gx, gy)
lap = cv2.Laplacian(sup, cv2.CV_64F, ksize=3)

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a, (t, im) in zip(ax, [("|Gx| horizontal", np.abs(gx)), ("|Gy| vertical", np.abs(gy)),
                           ("magnitude Sobel", mag), ("Laplaciano", np.abs(lap))]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()
print("Gx realça bordas VERTICAIS (derivada em x); Gy realça bordas HORIZONTAIS.")

## 7. SciPy `ndimage` — o mesmo filtro, outra biblioteca

O `scipy.ndimage` trabalha em N dimensões: o mesmo código serve para uma tomografia 3D.

In [ ]:
from scipy import ndimage
import time

t0 = time.perf_counter(); a = cv2.GaussianBlur(img, (7,7), 1.5);        t_cv = time.perf_counter()-t0
t0 = time.perf_counter(); b = ndimage.gaussian_filter(img, sigma=1.5);  t_sp = time.perf_counter()-t0

print(f"OpenCV        : {t_cv*1000:7.2f} ms")
print(f"scipy.ndimage : {t_sp*1000:7.2f} ms")
print(f"diferença média absoluta entre os dois resultados: {np.abs(a.astype(int)-b.astype(int)).mean():.2f} níveis")
print("\nMesma operação matemática, implementações e bordas tratadas de forma diferente.")

## 8. Segmentação dos campos pulmonares com Otsu

In [ ]:
lim, binaria = cv2.threshold(sup, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
print("limiar escolhido automaticamente por Otsu:", lim)

k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
aberta  = cv2.morphologyEx(binaria, cv2.MORPH_OPEN,  k)
fechada = cv2.morphologyEx(aberta,  cv2.MORPH_CLOSE, k)

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a, (t, im) in zip(ax, [("suavizada", sup), (f"Otsu (limiar={int(lim)})", binaria),
                           ("abertura", aberta), ("+ fechamento", fechada)]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

> O resultado ainda inclui o fundo fora do tórax. Precisamos de um recorte do campo de exame e da seleção dos dois maiores componentes.

In [ ]:
# === Funções do pipeline PulmoScan (reutilizadas em todos os laboratórios) ===
import cv2, numpy as np

ROI = (36, 220)          # recorte do campo de exame
K_TOPHAT = 15            # elemento estruturante do realce de nódulo

def suavizar(img):
    """Mediana 3x3 (sal-e-pimenta) seguida de Gaussiano 3x3 (ruído de detector)."""
    return cv2.GaussianBlur(cv2.medianBlur(img, 3), (3, 3), 0)

def mascara_pulmao(sup):
    """Recorte -> Otsu invertido -> morfologia -> os 2 maiores componentes."""
    lo, hi = ROI
    roi = np.zeros(sup.shape, np.uint8); roi[lo:hi, lo:hi] = 255
    _, b = cv2.threshold(sup, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    b = cv2.bitwise_and(b, roi)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, k)
    b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k)
    n, lab, st, _ = cv2.connectedComponentsWithStats(b, 8)
    out = np.zeros_like(b)
    for _, i in sorted(((st[i, cv2.CC_STAT_AREA], i) for i in range(1, n)), reverse=True)[:2]:
        out[lab == i] = 255
    return out

def realce_nodulo(sup):
    """Top-hat branco: mantém apenas estruturas claras MENORES que o elemento."""
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (K_TOPHAT, K_TOPHAT))
    return cv2.GaussianBlur(cv2.morphologyEx(sup, cv2.MORPH_TOPHAT, k), (5, 5), 0)

ATRIBUTOS = ["tophat_max","tophat_p999","tophat_media","tophat_dp","area_pulmao",
             "n_blobs","maior_blob","soma_blobs","brilho_medio","contraste_img"]

def atributos(img):
    sup = suavizar(img); m = mascara_pulmao(sup); th = realce_nodulo(sup)
    dentro = th[m > 0] if (m > 0).any() else th.ravel()
    resp = (th.astype(np.float32) * (m > 0)).astype(np.uint8)
    lim = int(max(dentro.mean() + 3*dentro.std(), 8.0))
    _, b = cv2.threshold(resp, lim, 255, cv2.THRESH_BINARY)
    n, _, st, _ = cv2.connectedComponentsWithStats(b, 8)
    ar = st[1:, cv2.CC_STAT_AREA] if n > 1 else np.array([0])
    return [float(dentro.max()), float(np.percentile(dentro, 99.9)), float(dentro.mean()),
            float(dentro.std()), float((m > 0).mean()), float(max(n-1, 0)),
            float(ar.max()), float(ar.sum()), float(img.mean()), float(img.std())]

print("pipeline carregado:", len(ATRIBUTOS), "atributos")

In [ ]:
m = mascara_pulmao(sup)
print("fração da imagem ocupada pelos campos pulmonares:", round((m>0).mean(), 3))

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("original")
ax[1].imshow(m, cmap="gray");   ax[1].set_title("máscara final (2 componentes)")
sobrep = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB); sobrep[m>0] = (0.6*sobrep[m>0] + np.array([102,0,0])).astype(np.uint8)
ax[2].imshow(sobrep); ax[2].set_title("sobreposição")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

**Responda:** por que descartar os componentes que tocam a borda da imagem não funcionou aqui, e o recorte do campo de exame funcionou?



## 9. Realce do nódulo com top-hat

**A regra que decide tudo:** o top-hat branco só enxerga estruturas claras **menores** que o elemento estruturante. Um elemento pequeno demais não vê o nódulo — ele passa 'inteiro' pela abertura e sobra zero.

In [ ]:
alvo = meta[(meta.rotulo==1) & (meta.faixa_etaria=="idoso")].arquivo.iloc[0]
com  = suavizar(carregar("pulmoscan/" + alvo))
sem  = suavizar(carregar("pulmoscan/" + meta[meta.rotulo==0].arquivo.iloc[0]))
mk   = mascara_pulmao(com)

print("resposta máxima do top-hat dentro dos pulmões, por tamanho do elemento:\n")
print(f"{'elemento':>10} {'com nódulo':>12} {'sem nódulo':>12}")
for ks in (5, 9, 15, 21, 27, 35):
    ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ks,ks))
    a = cv2.morphologyEx(com, cv2.MORPH_TOPHAT, ke)[mk>0].max()
    b = cv2.morphologyEx(sem, cv2.MORPH_TOPHAT, mascara_pulmao(sem)>0 if False else ke)[mascara_pulmao(sem)>0].max()
    print(f"{ks:>7}x{ks:<3} {a:>12} {b:>12}")

In [ ]:
th = realce_nodulo(com)
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(com, cmap="gray"); ax[0].set_title("suavizada")
ax[1].imshow(th, cmap="hot");   ax[1].set_title(f"top-hat (elemento {K_TOPHAT}x{K_TOPHAT})")
ax[2].imshow(th*(mk>0), cmap="hot"); ax[2].set_title("top-hat restrito aos pulmões")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 10. Extração dos atributos de todas as 240 imagens

In [ ]:
X = pd.DataFrame([atributos(carregar("pulmoscan/"+a)) for a in meta.arquivo], columns=ATRIBUTOS)
print(X.describe().round(2).to_string())
X.to_csv("atributos.csv", index=False)
print("\nsalvo: atributos.csv", X.shape)

## 11. Checklist de qualidade do pré-processamento

Rode esta célula ao final de **todo** módulo de pré-processamento que você escrever.

In [ ]:
ok = True
def check(nome, cond, det=""):
    global ok; ok &= bool(cond)
    print(f"[{'OK  ' if cond else 'FALHA'}] {nome} {det}")

check("1. todas as imagens foram lidas",      len(X) == len(meta), f"({len(X)}/{len(meta)})")
check("2. sem valores faltantes",             X.isna().sum().sum() == 0)
check("3. sem atributo constante",            (X.nunique() > 1).all())
check("4. máscara plausível em toda imagem",  X.area_pulmao.between(0.10, 0.35).all(),
      f"(faixa observada {X.area_pulmao.min():.3f}–{X.area_pulmao.max():.3f})")
check("5. sem valor infinito",                np.isfinite(X.values).all())
check("6. dimensionalidade controlada",       X.shape[1] <= 50, f"({X.shape[1]} atributos)")
print("\n>>> PRÉ-PROCESSAMENTO APROVADO" if ok else "\n>>> CORRIJA OS ITENS ACIMA")

## 12. SUA VEZ

Adicione um atributo novo ao vetor: a **excentricidade do maior componente claro** dentro dos pulmões (use `cv2.fitEllipse` sobre o maior contorno). Nódulos tendem a ser redondos; vasos, alongados.

In [ ]:
# SUA VEZ
# 1. encontre os contornos da imagem binarizada do top-hat (cv2.findContours)
# 2. selecione o maior contorno com pelo menos 5 pontos (exigência do fitEllipse)
# 3. calcule a excentricidade a partir dos eixos da elipse ajustada
# 4. devolva 0.0 quando não houver contorno válido


---
## Entrega

Poste o **link do notebook** no AVA, com todas as células de texto respondidas e o atributo novo implementado. Prazo no AVA da disciplina. Conta como atividade de N1 e alimenta o **Artefato 2**.